In [1]:
# Cài đặt các thư viện bắt buộc cho Pipeline và ép cập nhật torchao
!pip install -q -U "torchao>0.16.0"
!pip install -q open_clip_torch
!pip install -q evaluate
!pip install -q rouge_score
!pip install -q bert_score
!pip install -q datasets
!pip install -q accelerate
!pip install -q scikit-learn
!pip install -q transformers qwen-vl-utils
!pip install evaluate rouge_score bert_score scikit-learn
!pip install -q bitsandbytes accelerate
print("✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 60.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.7 MB/s eta 0:00:00:00:0100:01
✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!


In [2]:
import os
import gc
import copy
import time
import torch
import re
import numpy as np
import pandas as pd
from PIL import Image
from datasets import load_dataset
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
import evaluate
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# ==========================================
# 0. THIẾT LẬP MÔI TRƯỜNG KAGGLE
# ==========================================
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Tên Run được đổi thành c5 (5 Clients)
runName = "FedMVQA_VQARAD_Qwen2VL2B_LoRA_a10_c5_R32_E2_DetailedLog"

kaggleBaseDir = "/kaggle/working/Fed-MVQA-Project"
dataDir = os.path.join(kaggleBaseDir, "data", "vqarad")
logDir = os.path.join(kaggleBaseDir, "logs")
checkpointDir = os.path.join(kaggleBaseDir, "checkpoints", runName)

os.makedirs(dataDir, exist_ok=True)
os.makedirs(logDir, exist_ok=True)
os.makedirs(checkpointDir, exist_ok=True)

logPath = os.path.join(logDir, f"{runName}_log.csv")
checkpointPath = os.path.join(checkpointDir, "best_fed_mvqa_model_c5.pt")

# ==========================================
# 1. CÔNG CỤ QUẢN LÝ BỘ NHỚ VÀ ĐO LƯỜNG
# ==========================================
def clearMemory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def getPeakReservedMemory():
    if torch.cuda.is_available():
        peak_gb = torch.cuda.max_memory_reserved() / (1024 ** 3)
        torch.cuda.reset_peak_memory_stats()
        return f"{peak_gb:.2f} GB"
    return "0.00 GB"

def calculateTrainableParameters(model):
    trainableParams = 0
    allParams = 0
    for _, param in model.named_parameters():
        allParams += param.numel()
        if param.requires_grad:
            trainableParams += param.numel()
    percentage = 100 * trainableParams / allParams
    return trainableParams, allParams, percentage

def printStaticMemoryReport(totalParams, trainableParams):
    model_weights_gb = (totalParams * 2) / (1024**3)
    gradients_gb = (trainableParams * 2) / (1024**3)
    optimizer_states_gb = (trainableParams * 2) / (1024**3) 
    total_static_gb = model_weights_gb + gradients_gb + optimizer_states_gb
    
    print("\n[BÁO CÁO CẤP PHÁT BỘ NHỚ TĨNH (STATIC VRAM ALLOCATION)]")
    print(f"  + Trọng số Base Model (FP16) : {model_weights_gb:.4f} GB")
    print(f"  + Đạo hàm Gradients (FP16)   : {gradients_gb:.4f} GB")
    print(f"  + Optimizer (8-bit Adam)     : {optimizer_states_gb:.4f} GB")
    print(f"  => TỔNG VRAM TĨNH YÊU CẦU    : {total_static_gb:.4f} GB")
    return total_static_gb

def calculateCommunicationMetrics(loraWeights, totalParams, numClients):
    peftBytes = sum(tensor.numel() * tensor.element_size() for key, tensor in loraWeights.items())
    peftRoundCostMb = (peftBytes * 2 * numClients) / (1024 * 1024) 
    
    fullModelBytes = totalParams * 2 
    fullRoundCostMb = (fullModelBytes * 2 * numClients) / (1024 * 1024)
    
    percentageReduction = ((fullRoundCostMb - peftRoundCostMb) / fullRoundCostMb) * 100
    return peftRoundCostMb, fullRoundCostMb, percentageReduction

# ==========================================
# 2. KHỞI TẠO MÔ HÌNH TOÀN CỤC (LORA)
# ==========================================
print("--- KHỞI TẠO HẠ TẦNG FED-MVQA (LORA - 5 CLIENTS, QWEN2-VL-2B) ---")
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    min_pixels=256 * 28 * 28,
    max_pixels=256 * 28 * 28 
)

baseModel = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16, 
    device_map={"": 0}, 
    attn_implementation="sdpa"
)

baseModel.config.use_cache = False
baseModel.gradient_checkpointing_enable()

loraConfig = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

globalPeftModel = get_peft_model(baseModel, loraConfig)
trainableParams, totalModelParams, trainPct = calculateTrainableParameters(globalPeftModel)
print(f"[Metrics] Trainable Params (LoRA): {trainableParams:,} (~{trainPct:.3f}%)")

static_vram_gb = printStaticMemoryReport(totalModelParams, trainableParams)

def extractLoraWeights(model):
    return {k: v.cpu().clone() for k, v in model.state_dict().items() if "lora" in k}

globalWeights = extractLoraWeights(globalPeftModel)

# ==========================================
# 3. CHUẨN BỊ VÀ PHÂN MẢNH DỮ LIỆU VQA-RAD (ALPHA = 1.0)
# ==========================================
print("\n--- CHUẨN BỊ DỮ LIỆU VQA-RAD (GẦN IID α=1.0, 5 CLIENTS) ---")
dataset = load_dataset("flaviagiammarino/vqa-rad", cache_dir=dataDir)

shuffledTrain = dataset["train"].shuffle(seed=42)
testDataset = dataset["test"]

def partitionDataNonIidDirichlet(datasetInput, numClients=5, alpha=1.0):
    labels = [str(item["answer"]).strip().lower() for item in datasetInput]
    uniqueLabels = list(set(labels))
    clientIndices = {i: [] for i in range(numClients)}
    for label in uniqueLabels:
        labelIdxs = [i for i, l in enumerate(labels) if l == label]
        np.random.shuffle(labelIdxs)
        proportions = np.random.dirichlet([alpha] * numClients)
        splits = np.split(labelIdxs, (np.cumsum(proportions)[:-1] * len(labelIdxs)).astype(int))
        for i in range(numClients):
            clientIndices[i].extend(splits[i].tolist())
            
    clientDatasets = []
    for i in range(numClients):
        np.random.shuffle(clientIndices[i])
        clientDatasets.append(datasetInput.select(clientIndices[i]))
    return clientDatasets

# Chia cho 5 Clients
clientDatasets = partitionDataNonIidDirichlet(shuffledTrain, numClients=5, alpha=1.0)
datasetClientA = clientDatasets[0]
datasetClientB = clientDatasets[1]
datasetClientC = clientDatasets[2]
datasetClientD = clientDatasets[3]
datasetClientE = clientDatasets[4]

print(f"Tổng số mẫu huấn luyện : {len(shuffledTrain)} mẫu")
print(f"  + Dữ liệu Client A: {len(datasetClientA)} mẫu")
print(f"  + Dữ liệu Client B: {len(datasetClientB)} mẫu")
print(f"  + Dữ liệu Client C: {len(datasetClientC)} mẫu")
print(f"  + Dữ liệu Client D: {len(datasetClientD)} mẫu")
print(f"  + Dữ liệu Client E: {len(datasetClientE)} mẫu")
print(f"Tổng số mẫu Global Test: {len(testDataset)} mẫu")

headerTokens = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
headerLen = len(headerTokens)

systemInstruction = (
    "You are an expert medical diagnostic assistant. Your task is to answer clinical questions "
    "based EXCLUSIVELY on the provided medical image.\n"
    "Strictly follow these rules:\n"
    "1. Analyze the provided [Target Case Image] and the [Question].\n"
    "2. Provide a concise and definitive answer (1-3 words if possible)."
)

def customCollateFn(batch):
    texts, images = [], []
    for item in batch:
        img = item["image"].convert("RGB")
        q = item["question"]
        a = str(item["answer"]).strip()
        
        messages = [
            {"role": "system", "content": [{"type": "text", "text": systemInstruction}]},
            {"role": "user", "content": [
                {"type": "text", "text": "[Target Case Image]\n"},
                {"type": "image", "image": img}, 
                {"type": "text", "text": f"\nCurrent Question: {q}\n\nBased on the visual evidence, what is the correct answer for the current case?\nFinal Answer:"}
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": a}]}
        ]
        textPrompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(textPrompt)
        images.append(img)

    inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = inputs["input_ids"].clone()
    for i in range(labels.shape[0]):
        tokenIds = labels[i].tolist()
        if processor.tokenizer.pad_token_id is not None:
            labels[i][labels[i] == processor.tokenizer.pad_token_id] = -100
        matchIdx = -1
        for j in range(len(tokenIds) - headerLen + 1):
            if tokenIds[j:j+headerLen] == headerTokens:
                matchIdx = j + headerLen 
                break
        if matchIdx != -1:
            labels[i, :matchIdx] = -100 
    inputs["labels"] = labels
    return inputs

batchSize = 2
trainLoaderA = DataLoader(datasetClientA, batch_size=batchSize, shuffle=True, collate_fn=customCollateFn, num_workers=0, pin_memory=True)
trainLoaderB = DataLoader(datasetClientB, batch_size=batchSize, shuffle=True, collate_fn=customCollateFn, num_workers=0, pin_memory=True)
trainLoaderC = DataLoader(datasetClientC, batch_size=batchSize, shuffle=True, collate_fn=customCollateFn, num_workers=0, pin_memory=True)
trainLoaderD = DataLoader(datasetClientD, batch_size=batchSize, shuffle=True, collate_fn=customCollateFn, num_workers=0, pin_memory=True)
trainLoaderE = DataLoader(datasetClientE, batch_size=batchSize, shuffle=True, collate_fn=customCollateFn, num_workers=0, pin_memory=True)

# ==========================================
# 4. HÀM HUẤN LUYỆN CỤC BỘ 
# ==========================================
def trainLocalClient(model, dataLoader, clientId, epochs=2, learningRate=5e-5): 
    model.train()
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=learningRate, weight_decay=0.01)
    accumulationSteps = 4 
    
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs * len(dataLoader) // accumulationSteps)
    
    print(f"\n[Client {clientId}] Bắt đầu huấn luyện (Samples: {len(dataLoader.dataset)}, LR: {learningRate:.2e})...")
    startTime = time.time()
    finalLoss = 0
    
    for epoch in range(epochs):
        totalLoss = 0
        optimizer.zero_grad()
        
        progressLoop = tqdm(dataLoader, desc=f"   -> Epoch {epoch + 1}/{epochs}", leave=True, dynamic_ncols=True)
        
        for step, batch in enumerate(progressLoop):
            batch = {k: v.to(model.device) for k, v in batch.items()}
            
            with torch.amp.autocast("cuda"):
                outputs = model(**batch)
                loss = outputs.loss / accumulationSteps
            
            loss.backward()
            
            if (step + 1) % accumulationSteps == 0 or (step + 1) == len(dataLoader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                
            stepLoss = loss.item() * accumulationSteps
            totalLoss += stepLoss
            
            currentLr = scheduler.get_last_lr()[0]
            progressLoop.set_postfix(Loss=f"{stepLoss:.4f}", LR=f"{currentLr:.2e}")
            
            del outputs, loss, batch
            
        avgLoss = totalLoss / len(dataLoader)
        finalLoss = avgLoss
        print(f"      Epoch {epoch + 1} hoàn tất | Avg Loss: {avgLoss:.4f}")
    
    trainDuration = time.time() - startTime
    peakVram = getPeakReservedMemory()
    return extractLoraWeights(model), trainDuration, finalLoss, peakVram

# ==========================================
# 5. THUẬT TOÁN FEDERATED AVERAGING
# ==========================================
def simpleFederatedAveraging(globalWeightsInput, clientWeightsList, dataSizes):
    totalSamples = sum(dataSizes)
    newGlobalWeights = {} 
    
    keys = list(globalWeightsInput.keys())
    
    for key in keys:
        if "lora" in key:
            weightedSum = torch.zeros_like(globalWeightsInput[key], dtype=torch.float32, device="cpu")
            for i in range(len(clientWeightsList)):
                weightRatio = dataSizes[i] / totalSamples
                weightedSum += clientWeightsList[i][key].float() * weightRatio
            newGlobalWeights[key] = weightedSum.to(globalWeightsInput[key].dtype)
            
        for i in range(len(clientWeightsList)):
            if key in clientWeightsList[i]:
                del clientWeightsList[i][key]
                
    gc.collect()
    return newGlobalWeights

# ==========================================
# 6. HÀM ĐÁNH GIÁ
# ==========================================
def cleanOpenAnswer(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    stopwords = {'a', 'an', 'the', 'is', 'are', 'was', 'were', 'it', 'there', 'shows', 'showing', 'of', 'in', 'on', 'to'}
    words = text.split()
    words = [w for w in words if w not in stopwords]
    return " ".join(words)

def evaluateGlobalModel(model, datasetInput, processorInput):
    model.eval()
    print(f"\n [Server] Đang đánh giá trên tập Test ({len(datasetInput)} mẫu)...")
    
    openPreds, openRefs, closePreds, closeRefs = [], [], [], []
    overallCorrect, inferenceTimes = 0, []
    
    yesKeywords = ['yes', 'correct', 'true', 'positive', 'present', 'visible']
    noKeywords = ['no', 'incorrect', 'false', 'negative', 'absent', 'normal', 'none']
    
    evalLoop = tqdm(datasetInput, desc="Evaluating", leave=False, dynamic_ncols=True)
    
    for item in evalLoop:
        img = item["image"].convert("RGB")
        q = item["question"]
        gtAnswer = str(item["answer"]).strip().lower()
        ansType = str(item.get("answer_type", "")).strip().upper()
        if ansType == "": ansType = "CLOSED" if gtAnswer in ["yes", "no"] else "OPEN"
        
        messages = [
            {"role": "system", "content": [{"type": "text", "text": systemInstruction}]},
            {"role": "user", "content": [
                {"type": "text", "text": "[Target Case Image]\n"},
                {"type": "image", "image": img}, 
                {"type": "text", "text": f"\nCurrent Question: {q}\n\nBased on the visual evidence, what is the correct answer for the current case?\nFinal Answer:"}
            ]}
        ]
        
        textPrompt = processorInput.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processorInput(text=[textPrompt], images=[img], return_tensors="pt", padding=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        startTime = time.time()
        with torch.no_grad():
            generatedIds = model.generate(**inputs, max_new_tokens=15, do_sample=False)
        inferenceTimes.append(time.time() - startTime)
        
        generatedIdsTrimmed = [outIds[len(inIds):] for inIds, outIds in zip(inputs["input_ids"], generatedIds)]
        aiAnswerRaw = processorInput.batch_decode(generatedIdsTrimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip().lower()
        aiAnswer = aiAnswerRaw.replace("final answer:", "").strip()
        
        if ansType == "CLOSED":
            hasYes = any(word in aiAnswer for word in yesKeywords)
            hasNo = any(word in aiAnswer for word in noKeywords)
            
            if hasYes and not hasNo: predClean = 'yes'
            elif hasNo and not hasYes: predClean = 'no'
            else:
                firstYesPos = aiAnswer.find('yes') if 'yes' in aiAnswer else 9999
                firstNoPos = aiAnswer.find('no') if 'no' in aiAnswer else 9999
                predClean = 'yes' if firstYesPos < firstNoPos else ('no' if firstNoPos < firstYesPos else 'unknown')
                
            gtClean = 'yes' if 'yes' in gtAnswer else ('no' if 'no' in gtAnswer else gtAnswer)
            closePreds.append(predClean); closeRefs.append(gtClean)
            if predClean == gtClean: overallCorrect += 1
        else:
            aiAnswerClean = cleanOpenAnswer(aiAnswer)
            gtAnswerClean = cleanOpenAnswer(gtAnswer)
            openPreds.append(aiAnswerClean); openRefs.append(gtAnswerClean)
            if aiAnswerClean == gtAnswerClean: overallCorrect += 1
            
        del inputs, generatedIds, generatedIdsTrimmed
        torch.cuda.empty_cache()

    res = {"overallAccuracy": overallCorrect / len(datasetInput) if len(datasetInput) > 0 else 0, "avgInferenceTime": sum(inferenceTimes) / len(inferenceTimes)}
    if closeRefs: res["Closed"] = {"ACC": accuracy_score(closeRefs, closePreds), "F1": f1_score(closeRefs, closePreds, average='weighted', zero_division=0), "count": len(closeRefs)}
    if openRefs:
        bertRes = evaluate.load("bertscore").compute(predictions=openPreds, references=openRefs, lang="en")
        res["Open"] = {"BLEU-1": evaluate.load("bleu").compute(predictions=openPreds, references=[[r] for r in openRefs], max_order=1)["bleu"], "ROUGE-L": evaluate.load("rouge").compute(predictions=openPreds, references=[[r] for r in openRefs])["rougeL"], "BERTScore": sum(bertRes["f1"]) / len(bertRes["f1"]), "count": len(openRefs)}
    return res

# ==========================================
# 7. VÒNG LẶP HỌC LIÊN KẾT CHÍNH (5 CLIENTS)
# ==========================================
maxRounds = 1  
dataSizes = [len(datasetClientA), len(datasetClientB), len(datasetClientC), len(datasetClientD), len(datasetClientE)]
cumulativePeftGb = 0.0
performanceLog = []
peakVramRecorded = "0.00 GB"

bestLoss = float('inf')
patienceCounter = 0
patienceLimit = 2 

print(f"\n=== BẮT ĐẦU HUẤN LUYỆN FED-MVQA ({maxRounds} ROUNDS, 2 EPOCHS/CLIENT, 5 CLIENTS, ALPHA=1.0) ===")

for roundIdx in range(1, maxRounds + 1):
    print(f"\n" + "="*50)
    print(f"--- COMMUNICATION ROUND {roundIdx} ---")
    print("="*50)
    latestTrainTimes = []
    
    # Client A
    globalPeftModel.load_state_dict(globalWeights, strict=False)
    localWeightsA, timeA, lossA, vramA = trainLocalClient(globalPeftModel, trainLoaderA, clientId="A", epochs=2, learningRate=5e-5)
    latestTrainTimes.append(timeA)
    clearMemory()
    
    # Client B
    globalPeftModel.load_state_dict(globalWeights, strict=False)
    localWeightsB, timeB, lossB, vramB = trainLocalClient(globalPeftModel, trainLoaderB, clientId="B", epochs=2, learningRate=5e-5)
    latestTrainTimes.append(timeB)
    clearMemory()

    # Client C
    globalPeftModel.load_state_dict(globalWeights, strict=False)
    localWeightsC, timeC, lossC, vramC = trainLocalClient(globalPeftModel, trainLoaderC, clientId="C", epochs=2, learningRate=5e-5)
    latestTrainTimes.append(timeC)
    clearMemory()
    
    # Client D
    globalPeftModel.load_state_dict(globalWeights, strict=False)
    localWeightsD, timeD, lossD, vramD = trainLocalClient(globalPeftModel, trainLoaderD, clientId="D", epochs=2, learningRate=5e-5)
    latestTrainTimes.append(timeD)
    clearMemory()

    # Client E
    globalPeftModel.load_state_dict(globalWeights, strict=False)
    localWeightsE, timeE, lossE, vramE = trainLocalClient(globalPeftModel, trainLoaderE, clientId="E", epochs=2, learningRate=5e-5)
    latestTrainTimes.append(timeE)
    peakVramRecorded = vramE
    clearMemory()
    
    print("\n [Server] Đang tổng hợp trọng số LoRA (FedAvg - 5 Clients)...")
    globalWeights = simpleFederatedAveraging(globalWeights, [localWeightsA, localWeightsB, localWeightsC, localWeightsD, localWeightsE], dataSizes)
    
    globalPeftModel.load_state_dict(globalWeights, strict=False)
    evalResults = evaluateGlobalModel(globalPeftModel, testDataset, processor)
    
    roundAccuracy = evalResults["overallAccuracy"] * 100
    roundAccClosed = evalResults.get("Closed", {}).get("ACC", 0) * 100
    roundF1Closed = evalResults.get("Closed", {}).get("F1", 0) * 100
    roundBleu = evalResults.get("Open", {}).get("BLEU-1", 0)
    roundRouge = evalResults.get("Open", {}).get("ROUGE-L", 0)
    roundBert = evalResults.get("Open", {}).get("BERTScore", 0)
    
    peftRoundMb, fullRoundMb, reductionPct = calculateCommunicationMetrics(globalWeights, totalModelParams, numClients=5)
    cumulativePeftGb += (peftRoundMb / 1024)
    
    avgTrainingTime = sum(latestTrainTimes) / len(latestTrainTimes) if len(latestTrainTimes) > 0 else 0
    averageRoundLoss = (lossA + lossB + lossC + lossD + lossE) / 5
    
    roundLog = {
        "Round": roundIdx,
        "Avg Loss": round(averageRoundLoss, 4),
        "Overall Acc (%)": round(roundAccuracy, 2),
        "Closed Acc (%)": round(roundAccClosed, 2),
        "Closed F1 (%)": round(roundF1Closed, 2),
        "Open BLEU-1": round(roundBleu, 4),
        "Open ROUGE-L": round(roundRouge, 4),
        "Open BERTScore": round(roundBert, 4),
        "MB per Round": round(peftRoundMb, 2),
        "Avg Train Time (s)": round(avgTrainingTime, 2),
        "Peak VRAM": peakVramRecorded,
        "Cumul Data (GB)": round(cumulativePeftGb, 4)
    }
    performanceLog.append(roundLog)
    
    print(f"\n [KẾT QUẢ VÒNG {roundIdx}]")
    print(f"  + Loss Trung Bình  : {averageRoundLoss:.4f}")
    print(f"  + Overall Accuracy : {roundAccuracy:.2f}%")
    print(f"  + Closed Questions : Acc = {roundAccClosed:.2f}%, F1 = {roundF1Closed:.2f}%")
    print(f"  + Open Questions   : BLEU-1 = {roundBleu:.4f}, ROUGE-L = {roundRouge:.4f}, BERTScore = {roundBert:.4f}")
    print(f"  + Hiệu Suất        : Train = {avgTrainingTime:.2f}s | Static VRAM = {static_vram_gb:.4f} GB | Dynamic VRAM = {peakVramRecorded}")
    print(f"  + Truyền tải       : {peftRoundMb:.2f} MB/Round (Tổng: {cumulativePeftGb:.4f} GB)")

    pd.DataFrame(performanceLog).to_csv(logPath, index=False)
    
    if averageRoundLoss < bestLoss:
        bestLoss = averageRoundLoss
        patienceCounter = 0
        torch.save(globalWeights, checkpointPath)
        print(f"\n   -> [Cập nhật] Mô hình tốt nhất đã được lưu tại Round {roundIdx}.")
    else:
        patienceCounter += 1
        print(f"\n   -> [Cảnh báo] Loss không cải thiện. Patience: {patienceCounter}/{patienceLimit}")
        
    if patienceCounter >= patienceLimit:
        print(f"\n[HỘI TỤ] Hệ thống kích hoạt dừng sớm tại Vòng {roundIdx}.")
        break

# ==========================================
# 8. XUẤT BÁO CÁO TỔNG KẾT
# ==========================================
print("\n" + "="*80)
print(f" FINAL SUMMARY REPORT: FED-MVQA (VQA-RAD, {roundIdx} ROUNDS ĐÃ HOÀN TẤT, 5 CLIENTS) ")
print("="*80)

print("\n[1] PERFORMANCE & HARDWARE METRICS")
print(f"  + Trainable Parameters   : {trainableParams:,} (~{trainPct:.3f}%)")
print(f"  + Static Allocated VRAM  : {static_vram_gb:.4f} GB")
print(f"  + Dynamic Peak VRAM      : {peakVramRecorded} (Nhiễu do Padding)")
print(f"  + Average Inference Time : {evalResults['avgInferenceTime']:.4f} giây/ảnh")
print(f"  + Communication Savings  : Giảm {reductionPct:.2f}% băng thông so với FFT")

print("\n[2] CLINICAL EVALUATION METRICS (GLOBAL MODEL VÒNG CUỐI)")
if "Closed" in evalResults:
    print(f"  - CLOSED QUESTIONS ({evalResults['Closed']['count']} samples):")
    print(f"    * Accuracy : {evalResults['Closed']['ACC']*100:.2f}%")
    print(f"    * F1-Score : {evalResults['Closed']['F1']*100:.2f}%")
if "Open" in evalResults:
    print(f"  - OPEN QUESTIONS ({evalResults['Open']['count']} samples):")
    print(f"    * BLEU-1   : {evalResults['Open']['BLEU-1']:.4f}")
    print(f"    * ROUGE-L  : {evalResults['Open']['ROUGE-L']:.4f}")
    print(f"    * BERTScore: {evalResults['Open']['BERTScore']:.4f}")

print("\n[3] CONVERGENCE LOG CHI TIẾT")
commTable = pd.DataFrame(performanceLog)
print(commTable[['Round', 'Avg Loss', 'Overall Acc (%)', 'Closed Acc (%)', 'Open BLEU-1', 'MB per Round', 'Cumul Data (GB)', 'Avg Train Time (s)']].to_string(index=False))

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


--- KHỞI TẠO HẠ TẦNG FED-MVQA (LORA - 5 CLIENTS, QWEN2-VL-2B) ---


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

... (45 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



 [KẾT QUẢ VÒNG 1]
  + Loss Trung Bình  : 0.4792
  + Overall Accuracy : 47.23%
  + Closed Questions : Acc = 66.93%, F1 = 66.52%
  + Open Questions   : BLEU-1 = 0.2246, ROUGE-L = 0.3021, BERTScore = 0.8944
  + Hiệu Suất        : Train = 525.56s | Static VRAM = 4.3209 GB | Dynamic VRAM = 13.99 GB
  + Truyền tải       : 1408.75 MB/Round (Tổng: 1.3757 GB)

   -> [Cập nhật] Mô hình tốt nhất đã được lưu tại Round 1.

 FINAL SUMMARY REPORT: FED-MVQA (VQA-RAD, 1 ROUNDS ĐÃ HOÀN TẤT, 5 CLIENTS) 

[1] PERFORMANCE & HARDWARE METRICS
  + Trainable Parameters   : 36,929,536 (~1.644%)
  + Static Allocated VRAM  : 4.3209 GB
  + Dynamic Peak VRAM      : 13.99 GB (Nhiễu do Padding)
  + Average Inference Time : 0.3969 giây/ảnh
  + Communication Savings  : Giảm 96.71% băng thông so với FFT

[2] CLINICAL EVALUATION METRICS (GLOBAL MODEL VÒNG CUỐI)
  - CLOSED QUESTIONS (251 samples):
    * Accuracy : 66.93%
    * F1-Score : 66.52%
  - OPEN QUESTIONS (200 samples):
    * BLEU-1   : 0.2246
    * ROUGE-L  : 0.